# 🤟 Qyran v4 — Transformer + Holistic (90%+ Target)

**Что нового:**
- MediaPipe Holistic: руки + тело + лицо (255 признаков)
- 60 кадров вместо 30
- Transformer архитектура
- Начинаем с 10 жестов → потом добавляем

### Файлы на Google Drive (корень My Drive):
- `landmarks_holistic.zip`
- `annotations.csv`

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive'

for f in ['landmarks_holistic.zip', 'annotations.csv']:
    assert os.path.exists(f'{DRIVE}/{f}'), f'❌ {f} не найден!'
print('✅ Файлы найдены')

!unzip -q -o {DRIVE}/landmarks_holistic.zip -d /content/

train_n = len(os.listdir('/content/landmarks_holistic/train'))
test_n = len(os.listdir('/content/landmarks_holistic/test'))
print(f'✅ Train: {train_n} | Test: {test_n}')

## 2. Load Data — Top 10 Gestures

In [ ]:
import numpy as np
import pandas as pd
import json

df = pd.read_csv(f'{DRIVE}/annotations.csv', sep='\t')

# ========================================
# 🔧 НАСТРОЙКА: Количество жестов
# Измени TOP_N чтобы добавить больше жестов
# Этап 1: 10, Этап 2: 20, Этап 3: 30 ...
# ========================================
TOP_N = 10

SEQ_LEN = 60
FEATURES = 255

# Топ классов по количеству видео
top_classes = df[df['train']==True]['text'].value_counts().head(TOP_N).index.tolist()
label_map = {cls: i for i, cls in enumerate(top_classes)}
idx_to_label = {i: cls for cls, i in label_map.items()}
NUM_CLASSES = len(label_map)

def load_split(name, is_train):
    sdf = df[(df['train']==is_train) & (df['text'].isin(top_classes))]
    X, y, skip = [], [], 0
    for _, r in sdf.iterrows():
        p = f'/content/landmarks_holistic/{name}/{r["attachment_id"]}.npy'
        if os.path.exists(p):
            d = np.load(p)
            if d.shape == (SEQ_LEN, FEATURES):
                X.append(d)
                y.append(label_map[r['text']])
            else:
                skip += 1
        else:
            skip += 1
    print(f'  {name}: loaded {len(X)}, skipped {skip}')
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

X_train, y_train = load_split('train', True)
X_test, y_test = load_split('test', False)

print(f'\n✅ Classes: {NUM_CLASSES}')
print(f'✅ X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'\n📋 Жесты:')
for i, cls in enumerate(top_classes):
    train_cnt = (y_train == i).sum()
    test_cnt = (y_test == i).sum()
    print(f'  {i+1}. "{cls}" — train: {train_cnt}, test: {test_cnt}')

## 3. Data Augmentation

In [ ]:
import tensorflow as tf

class HolisticAugmentor(tf.keras.utils.Sequence):
    """Real-time augmentation for holistic landmarks."""
    def __init__(self, X, y, batch_size=32, augment=True):
        self.X, self.y = X, y
        self.batch_size = batch_size
        self.augment = augment
        self.indices = np.arange(len(X))
        self.on_epoch_end()
    
    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))
    
    def on_epoch_end(self):
        np.random.shuffle(self.indices)
    
    def __getitem__(self, idx):
        bi = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]
        X_b = self.X[bi].copy()
        y_b = self.y[bi]
        if self.augment:
            for i in range(len(X_b)):
                X_b[i] = self._aug(X_b[i])
        return X_b, y_b
    
    def _aug(self, seq):
        # Gaussian noise
        if np.random.random() < 0.8:
            seq += np.random.normal(0, np.random.uniform(0.003, 0.01), seq.shape).astype(np.float32)
        # Scale
        if np.random.random() < 0.7:
            seq *= np.random.uniform(0.88, 1.12)
        # Temporal shift
        if np.random.random() < 0.5:
            s = np.random.randint(-4, 5)
            if s != 0:
                seq = np.roll(seq, s, axis=0)
                if s > 0: seq[:s] = 0
                else: seq[s:] = 0
        # Speed perturbation
        if np.random.random() < 0.4:
            speed = np.random.uniform(0.8, 1.2)
            n = seq.shape[0]
            new_n = max(3, int(n * speed))
            idx_old = np.linspace(0, n-1, new_n).astype(int)
            stretched = seq[idx_old]
            idx_new = np.linspace(0, len(stretched)-1, n).astype(int)
            seq = stretched[idx_new]
        # Random frame dropout
        if np.random.random() < 0.3:
            n_drop = np.random.randint(1, 4)
            drop_idx = np.random.choice(seq.shape[0], n_drop, replace=False)
            seq[drop_idx] = 0
        return seq

train_gen = HolisticAugmentor(X_train, y_train, batch_size=32, augment=True)
val_gen = HolisticAugmentor(X_test, y_test, batch_size=32, augment=False)
print(f'✅ Augmentor: {len(train_gen)} batches/epoch')

## 4. Transformer Model

In [ ]:
from tensorflow.keras import layers, models

print(f'TF: {tf.__version__} | GPU: {tf.config.list_physical_devices("GPU")}')

class PositionalEncoding(layers.Layer):
    """Sinusoidal positional encoding."""
    def __init__(self, max_len, d_model, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.d_model = d_model
    
    def build(self, input_shape):
        pos = np.arange(self.max_len)[:, np.newaxis]
        i = np.arange(self.d_model)[np.newaxis, :]
        angle = pos / np.power(10000, (2 * (i // 2)) / self.d_model)
        pe = np.zeros((self.max_len, self.d_model), dtype=np.float32)
        pe[:, 0::2] = np.sin(angle[:, 0::2])
        pe[:, 1::2] = np.cos(angle[:, 1::2])
        self.pe = tf.constant(pe[np.newaxis, :, :])  # (1, max_len, d_model)
        super().build(input_shape)
    
    def call(self, x):
        return x + self.pe[:, :tf.shape(x)[1], :]
    
    def get_config(self):
        config = super().get_config()
        config.update({'max_len': self.max_len, 'd_model': self.d_model})
        return config


class TransformerBlock(layers.Layer):
    """Single transformer encoder block."""
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.dropout_rate = dropout
    
    def build(self, input_shape):
        self.att = layers.MultiHeadAttention(
            num_heads=self.num_heads, key_dim=self.d_model // self.num_heads
        )
        self.ffn = tf.keras.Sequential([
            layers.Dense(self.ff_dim, activation='gelu'),
            layers.Dropout(self.dropout_rate),
            layers.Dense(self.d_model),
        ])
        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(self.dropout_rate)
        self.drop2 = layers.Dropout(self.dropout_rate)
        super().build(input_shape)
    
    def call(self, x, training=False):
        # Multi-head self-attention + residual
        attn = self.att(x, x, training=training)
        attn = self.drop1(attn, training=training)
        x = self.ln1(x + attn)
        # Feed-forward + residual
        ffn = self.ffn(x, training=training)
        ffn = self.drop2(ffn, training=training)
        return self.ln2(x + ffn)
    
    def get_config(self):
        config = super().get_config()
        config.update({
            'd_model': self.d_model, 'num_heads': self.num_heads,
            'ff_dim': self.ff_dim, 'dropout': self.dropout_rate
        })
        return config


def build_model(seq_len, features, num_classes):
    D_MODEL = 128
    NUM_HEADS = 8
    FF_DIM = 256
    NUM_BLOCKS = 4
    DROPOUT = 0.2
    
    inp = layers.Input(shape=(seq_len, features), name='input_landmarks')
    
    # Project to d_model
    x = layers.Dense(D_MODEL, name='projection')(inp)
    x = layers.BatchNormalization(name='bn_proj')(x)
    
    # Positional encoding
    x = PositionalEncoding(seq_len, D_MODEL, name='pos_encoding')(x)
    x = layers.Dropout(DROPOUT, name='drop_pos')(x)
    
    # Transformer blocks
    for i in range(NUM_BLOCKS):
        x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT, name=f'transformer_{i}')(x)
    
    # Global pooling (both avg and max for richer representation)
    avg_pool = layers.GlobalAveragePooling1D(name='gap')(x)
    max_pool = layers.GlobalMaxPooling1D(name='gmp')(x)
    x = layers.Concatenate(name='concat_pool')([avg_pool, max_pool])
    
    # Classification head
    x = layers.Dense(256, activation='gelu', name='dense_1')(x)
    x = layers.Dropout(0.3, name='drop_1')(x)
    x = layers.Dense(128, activation='gelu', name='dense_2')(x)
    x = layers.Dropout(0.2, name='drop_2')(x)
    out = layers.Dense(num_classes, activation='softmax', name='output')(x)
    
    model = models.Model(inp, out, name='qyran_transformer')
    
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model(SEQ_LEN, FEATURES, NUM_CLASSES)
model.summary()

# Verify unique names
names = [l.name for l in model.layers]
assert len(names) == len(set(names)), '❌ Duplicate names!'
print(f'\n✅ {len(names)} layers, all unique names')

## 5. Training

In [ ]:
from tensorflow.keras import callbacks as cb

callbacks_list = [
    cb.ModelCheckpoint(
        '/content/best_model.keras',
        monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
    ),
    cb.EarlyStopping(
        monitor='val_loss', patience=20, restore_best_weights=True, verbose=1
    ),
    cb.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1
    ),
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=150,
    callbacks=callbacks_list,
    verbose=1
)

print('\n🎯 Training complete!')

## 6. Results

In [ ]:
import matplotlib.pyplot as plt

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
y_probs = model.predict(X_test, verbose=0)
y_pred = y_probs.argmax(axis=1)
top5 = sum(y_test[i] in np.argsort(y_probs[i])[-5:] for i in range(len(y_test)))

print(f'🎯 Test Accuracy:  {test_acc*100:.2f}%')
print(f'🎯 Top-5 Accuracy: {top5/len(y_test)*100:.2f}%')
print(f'📉 Test Loss:      {test_loss:.4f}')

# Per-class accuracy
print(f'\n📊 Per-class accuracy:')
for i in range(NUM_CLASSES):
    mask = y_test == i
    if mask.sum() > 0:
        acc = (y_pred[mask] == i).mean()
        emoji = '✅' if acc >= 0.8 else '⚠️' if acc >= 0.5 else '❌'
        print(f'  {emoji} "{idx_to_label[i]}": {acc*100:.1f}% ({mask.sum()} samples)')

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['accuracy'], label='Train', lw=2)
ax1.plot(history.history['val_accuracy'], label='Val', lw=2)
ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(history.history['loss'], label='Train', lw=2)
ax2.plot(history.history['val_loss'], label='Val', lw=2)
ax2.set_title('Loss'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()

## 7. Save & Export to TF.js

In [ ]:
import shutil

SAVE_DIR = f'{DRIVE}/qyran_model_v4'
os.makedirs(SAVE_DIR, exist_ok=True)

model.save(f'{SAVE_DIR}/final_model.keras')
shutil.copy('/content/best_model.keras', f'{SAVE_DIR}/best_model.keras')
shutil.copy('/content/training_curves.png', f'{SAVE_DIR}/training_curves.png')

# Label map (only current top-N)
with open(f'{SAVE_DIR}/label_map.json', 'w', encoding='utf-8') as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)

config = {
    'num_classes': NUM_CLASSES,
    'sequence_length': SEQ_LEN,
    'features_per_frame': FEATURES,
    'top_n_classes': TOP_N,
    'test_accuracy': float(test_acc),
    'classes': top_classes,
    'architecture': 'Transformer (4 blocks, 8 heads, d=128)',
    'input_type': 'holistic (pose+face+hands)',
    'tf_version': tf.__version__
}
with open(f'{SAVE_DIR}/config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print(f'✅ Model saved to {SAVE_DIR}')

In [ ]:
!pip install tensorflowjs -q 2>/dev/null
import subprocess

TFJS_DIR = f'{SAVE_DIR}/tfjs_model'
SAVED_MODEL_DIR = '/content/saved_model_tmp'

# Export as SavedModel first
model.export(SAVED_MODEL_DIR)

# Convert to TF.js graph model
result = subprocess.run([
    'tensorflowjs_converter',
    '--input_format=tf_saved_model',
    '--output_format=tfjs_graph_model',
    '--signature_name=serving_default',
    '--saved_model_tags=serve',
    SAVED_MODEL_DIR, TFJS_DIR
], capture_output=True, text=True)

if result.returncode != 0:
    print('⚠️ Graph model failed, trying layers...')
    model.save('/content/tmp_model.keras')
    result = subprocess.run([
        'tensorflowjs_converter',
        '--input_format=keras',
        '--output_format=tfjs_layers_model',
        '/content/tmp_model.keras', TFJS_DIR
    ], capture_output=True, text=True)

if result.returncode == 0:
    shutil.copy(f'{SAVE_DIR}/label_map.json', f'{TFJS_DIR}/label_map.json')
    shutil.copy(f'{SAVE_DIR}/config.json', f'{TFJS_DIR}/config.json')
    print(f'\n✅ TF.js model saved!')
    for f in sorted(os.listdir(TFJS_DIR)):
        sz = os.path.getsize(f'{TFJS_DIR}/{f}')
        print(f'  📄 {f} ({sz/1024:.1f} KB)')
else:
    print(f'❌ Failed: {result.stderr[:300]}')

## 🎉 Готово!

### Чтобы добавить больше жестов:
1. Измени `TOP_N = 10` на `TOP_N = 20` в ячейке 2
2. Перезапусти все ячейки (Runtime → Run all)
3. Модель автоматически дообучится на большем количестве классов

### Скачай `tfjs_model/` и положи в `public/model/` веб-приложения